<a href="https://colab.research.google.com/github/hariprakashg/Data-Analysis-Projects/blob/main/llm_research_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import arxiv
from semanticscholar import SemanticScholar
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

In [ ]:


class ResearchAgent:
    def __init__(self):
        self.llm = OllamaLLM(model="llama3.2:3b")
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.arxiv_client = arxiv.Client()
        self.scholar = SemanticScholar()
        self.index = None

    def fetch_papers(self, query, max_results=20):
        results = []
        search = arxiv.Search(query=query, max_results=max_results)
        for paper in self.arxiv_client.results(search):
            results.append({
                'title': paper.title,
                'summary': paper.summary,
                'pdf_url': paper.pdf_url
            })
        return results

    def build_rag_index(self, papers):
        texts = [p['summary'] for p in papers]
        embeddings = self.embedder.encode(texts)
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings.astype('float32'))
        self.paper_texts = texts
        return len(papers)

    def find_relevant_papers(self, query, k=5):
        if self.index is None:
            return []
        query_emb = self.embedder.encode([query])
        distances, indices = self.index.search(query_emb.astype('float32'), k)
        return [{'text': self.paper_texts[i], 'dist': float(d)} for i,d in zip(indices[0], distances[0])]

    def generate_future_work(self, paper_title: str) -> str:
        papers = self.fetch_papers(paper_title)
        self.build_rag_index(papers)

        prompt = """
Paper: {title}
Recent papers: {context}

Generate "Future Work" section (250 words):
1. Research gaps
2. Emerging trends
3. Novel combinations
4. Evaluation methods
5. Interdisciplinary opportunities

Format: Markdown section with citations.
        """.format(
            title=paper_title,
            context=json.dumps(self.find_relevant_papers(paper_title))
        )

        chain = ChatPromptTemplate.from_template(prompt) | self.llm
        return chain.invoke({})




In [ ]:
agent = ResearchAgent()
future_work = agent.generate_future_work("federated learning privacy")
print(future_work)
